In [1]:
import os
import string
import numpy as np
from tqdm import tqdm
from pickle import dump, load

from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add
from tensorflow.keras.optimizers import Adam

In [2]:
def load_captions(filename):
    file = open(filename, 'r')
    text = file.read()
    file.close()
    return text

def create_mapping(doc):
    mapping = {}
    for line in doc.split('\n'):
        tokens = line.split(',')
        if len(tokens) < 2:
            continue
        image_id, caption = tokens[0], tokens[1]
        image_id = image_id.split('.')[0]

        if image_id not in mapping:
            mapping[image_id] = []
        mapping[image_id].append(caption)
    return mapping

doc = load_captions("captions.txt")
captions_mapping = create_mapping(doc)

In [3]:
def clean_captions(mapping):
    table = str.maketrans('', '', string.punctuation)
    for key, captions in mapping.items():
        for i in range(len(captions)):
            caption = captions[i]
            caption = caption.lower()
            caption = caption.translate(table)
            caption = caption.split()
            caption = [word for word in caption if len(word) > 1]
            caption = ' '.join(caption)
            captions[i] = 'startseq ' + caption + ' endseq'

clean_captions(captions_mapping)

In [4]:
all_captions = []
for key in captions_mapping:
    for caption in captions_mapping[key]:
        all_captions.append(caption)

tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)

vocab_size = len(tokenizer.word_index) + 1
max_length = max(len(caption.split()) for caption in all_captions)

print("Vocabulary Size:", vocab_size)
print("Max Caption Length:", max_length)

Vocabulary Size: 1840
Max Caption Length: 23


In [14]:
model = InceptionV3(weights='imagenet')
model = Model(model.input, model.layers[-2].output)

def extract_features(directory):
    features = {}

    for img_name in tqdm(os.listdir(directory)):

        # Skip hidden files and non-image files
        if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        img_path = os.path.join(directory, img_name)

        image = load_img(img_path, target_size=(299, 299))
        image = img_to_array(image)
        image = np.expand_dims(image, axis=0)
        image = preprocess_input(image)

        feature = model.predict(image, verbose=0)

        image_id = img_name.split('.')[0]
        features[image_id] = feature

    return features

In [15]:
features = extract_features("Images")

100%|██████████| 3346/3346 [10:43<00:00,  5.20it/s]


In [16]:
X1, X2, y = [], [], []

for key, captions in captions_mapping.items():
    if key not in features:
        continue   # Skip missing images

    photo = features[key]
    in_img, in_seq, out_word = create_sequences(
        tokenizer, max_length, captions, photo, vocab_size
    )

    X1.extend(in_img)
    X2.extend(in_seq)
    y.extend(out_word)

In [17]:
# Image Model
inputs1 = Input(shape=(2048,))
fe1 = Dropout(0.5)(inputs1)
fe2 = Dense(256, activation='relu')(fe1)

# Text Model
inputs2 = Input(shape=(max_length,))
se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
se2 = Dropout(0.5)(se1)
se3 = LSTM(256)(se2)

# Merge
decoder1 = add([fe2, se3])
decoder2 = Dense(256, activation='relu')(decoder1)
outputs = Dense(vocab_size, activation='softmax')(decoder2)

model = Model(inputs=[inputs1, inputs2], outputs=outputs)

model.compile(loss='categorical_crossentropy',
              optimizer=Adam(learning_rate=0.001))

model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 23)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_6       │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 23, 256)   │    471,040 │ input_layer_7[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 2048)      │          0 │ input_layer_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 23, 256)   │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 23)        │          0 │ input_layer_7[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 256)       │    524,544 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 256)       │    525,312 │ dropout_3[0][0],  │
│                     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 256)       │          0 │ dense_3[0][0],    │
│                     │                   │            │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 256)       │     65,792 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1840)      │    472,880 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,059,568 (7.86 MB)

 Trainable params: 2,059,568 (7.86 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
X1, X2, y = [], [], []

for key in captions_mapping:
    if key not in features:
        continue

    photo = features[key][0]   # IMPORTANT FIX (remove extra dimension)

    for caption in captions_mapping[key]:
        seq = tokenizer.texts_to_sequences([caption])[0]

        for i in range(1, len(seq)):
            in_seq = seq[:i]
            out_seq = seq[i]

            in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
            out_seq = to_categorical(out_seq, num_classes=vocab_size)

            X1.append(photo)
            X2.append(in_seq)
            y.append(out_seq)

X1 = np.array(X1)
X2 = np.array(X2)
y = np.array(y)

In [22]:
print(X1.shape)
print(X2.shape)
print(y.shape)

(6092, 2048)
(6092, 23)
(6092, 1840)


In [24]:
model.fit([X1, X2], y, epochs=20, batch_size=64)

Epoch 1/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 12s 121ms/step - loss: 0.6324
Epoch 2/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 11s 118ms/step - loss: 0.5943
Epoch 3/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 11s 119ms/step - loss: 0.5381
Epoch 4/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 12s 119ms/step - loss: 0.5141
Epoch 5/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 11s 119ms/step - loss: 0.4782
Epoch 6/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 11s 119ms/step - loss: 0.4192
Epoch 7/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 11s 118ms/step - loss: 0.4008
Epoch 8/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 11s 118ms/step - loss: 0.3776
Epoch 9/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 11s 118ms/step - loss: 0.3660
Epoch 10/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 11s 118ms/step - loss: 0.3283
Epoch 11/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 11s 117ms/step - loss: 0.3157
Epoch 12/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 11s 118ms/step - loss: 0.3182
Epoch 13/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 11s 118ms/step - loss: 0.2876
Epoch 14/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 11s 118ms/step - loss: 0.2929
Epoch 15/20
96/96 ━━━━━━━━━━━━━━━━━━━━ 11s 

In [47]:
def generate_caption(model, tokenizer, photo, max_length):
    in_text = 'startseq'

    for i in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length)

        yhat = model.predict([photo, sequence], verbose=0)
        yhat = np.argmax(yhat)

        word = None
        for w, index in tokenizer.word_index.items():
            if index == yhat:
                word = w
                break

        if word is None:
            break

        in_text += ' ' + word

        if word == 'endseq':
            break

    return in_text

In [48]:
caption = generate_caption(model, tokenizer, photo, max_length)
print("Generated Caption:", caption)

Generated Caption: startseq bundled up endseq


In [41]:
def generate_caption(model, tokenizer, photo, max_length):
    in_text = 'startseq'

    for i in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length)

        yhat = model.predict([photo, sequence], verbose=0)
        yhat = np.argmax(yhat)

        word = None
        for w, index in tokenizer.word_index.items():
            if index == yhat:
                word = w
                break

        if word is None:
            break

        in_text += ' ' + word

        if word == 'endseq':
            break

    return in_text.replace('startseq', '').replace('endseq', '')

In [42]:
caption = generate_caption(model, tokenizer, photo, max_length)
print("Generated Caption:", caption)

Generated Caption:  bundled up 


In [43]:
photo = np.expand_dims(photo, axis=0)
print(photo)

[[[0.10859674 1.2652308  0.09501778 ... 0.45106328 0.9349948  0.20541611]]]


In [44]:
image_id = list(features.keys())[1]
photo = features[image_id]

print("Testing on image:", image_id)

caption = generate_caption(model, tokenizer, photo, max_length)
print(caption)

Testing on image: 1007320043_627395c3d8
 holding bag 


In [45]:
image_id = list(features.keys())[3]
photo = features[image_id]

print("Testing on image:", image_id)

caption = generate_caption(model, tokenizer, photo, max_length)
print(caption)

Testing on image: 101669240_b2d3e7f17b
 bundled up 


In [46]:
sequence = tokenizer.texts_to_sequences(['startseq'])[0]
sequence = pad_sequences([sequence], maxlen=max_length)

yhat = model.predict([photo, sequence])
print(np.argmax(yhat))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
646


In [49]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: nltk in c:\users\sandadi pranavi\anaconda3\lib\site-packages (3.9.1)



In [50]:
import nltk
from nltk.translate.bleu_score import corpus_bleu

In [51]:
def generate_caption(model, tokenizer, photo, max_length):
    in_text = 'startseq'
    
    for i in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length)

        yhat = model.predict([photo, sequence], verbose=0)
        yhat = np.argmax(yhat)

        word = None
        for w, index in tokenizer.word_index.items():
            if index == yhat:
                word = w
                break

        if word is None:
            break

        in_text += ' ' + word
        if word == 'endseq':
            break

    final_caption = in_text.split()
    final_caption = final_caption[1:-1]   # remove startseq & endseq
    return final_caption

In [52]:
actual, predicted = [], []

for key in captions_mapping:
    if key not in features:
        continue

    photo = features[key]
    
    y_pred = generate_caption(model, tokenizer, photo, max_length)
    predicted.append(y_pred)

    references = []
    for caption in captions_mapping[key]:
        ref = caption.split()
        ref = ref[1:-1]  # remove startseq & endseq
        references.append(ref)

    actual.append(references)

In [55]:
print("BLEU-1: %f" % corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0)))
print("BLEU-2: %f" % corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0)))
print("BLEU-3: %f" % corpus_bleu(actual, predicted, weights=(0.33, 0.33, 0.33, 0)))
print("BLEU-4: %f" % corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25)))

BLEU-1: 0.726715
BLEU-2: 0.681082
BLEU-3: 0.642764
BLEU-4: 0.594550


In [56]:
from nltk.translate.bleu_score import SmoothingFunction

smooth = SmoothingFunction().method1

print("BLEU-4:",
      corpus_bleu(actual, predicted,
                  weights=(0.25,0.25,0.25,0.25),
                  smoothing_function=smooth))

BLEU-4: 0.5945499915879484
